# Industrial surface defect segmentation
Sarthak Jain · 23f3000839 · DL & GenAI T3 2026

This registered notebook runs the frozen validation-selected model configuration. Training, three-model comparison and the inference ablation are linked below. The attached private assets contain checkpoints and an offline dependency wheel. Inference uses no downloads or additional labelled data.

- [Training experiment](https://www.kaggle.com/code/sarthakiitmbs/23f3000839-surface-defects-training-t32026)
- [Validation and error analysis](https://www.kaggle.com/code/sarthakiitmbs/23f3000839-surface-final-validation-t32026)
- [W&B runs](https://wandb.ai/23f3000839-iit-madras-foundation/23f3000839-t32026)

In [1]:
from pathlib import Path
import importlib.util, subprocess, sys, json, time
import numpy as np
import pandas as pd
import torch

ROOT = Path("/kaggle/input")
selections = list(ROOT.rglob("final_selection.json"))
assert len(selections) == 1, selections
ASSETS = selections[0].parent
if importlib.util.find_spec("segmentation_models_pytorch") is None:
    wheel = next(ASSETS.rglob("segmentation_models_pytorch-0.5.0-*.whl"))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheel)]
    )
torch.set_num_threads(4)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print(json.loads(selections[0].read_text()))


Processing /kaggle/input/23f3000839-surface-final-models-t32026/segmentation_models_pytorch-0.5.0-py3-none-any.whl
Device: cuda
{'variant': 'equal_mean_ensemble', 'model_names': ['scratch_unet', 'resnet34_unet', 'efficientnet_b0_fpn'], 'tta': False, 'forward_passes': 3, 'tune_assessment_score': 0.7076337834583714, 'audit_assessment_score': 0.683852163532016, 'audit_macro_pixel_f1': 0.5183344032313825, 'audit_pixel_accuracy': 0.9951685940926535, 'thresholds': [0.35, 0.35, 0.35, 0.35], 'min_areas': [32, 128, 128, 128], 'model': 'ensemble', 'selection_rule': 'highest tune score; prefer fewer forward passes within 0.002 of best'}


## Exact model definitions and preprocessing
The model definitions are extracted from the canonical training module. Every architecture produces four logits at the original 128 × 800 resolution. The shared prediction wrapper uses the same grayscale normalization, optional horizontal TTA, probability averaging and connected-component filter selected during validation.

In [2]:
%%writefile /kaggle/working/final_pipeline.py
"""Inference definitions extracted from the trained model source."""

import numpy as np
import torch
from torch import nn
from scipy import ndimage

CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
H, W = 128, 800


def encode_rle(mask):
    flat = mask.astype(np.uint8).ravel(order="F")
    changes = np.flatnonzero(np.diff(np.r_[0, flat, 0])) + 1
    changes[1::2] -= changes[::2]
    return " ".join(map(str, changes)) or " "


def remove_components(mask, min_area):
    if not min_area:
        return mask
    labels, n = ndimage.label(mask)
    sizes = np.bincount(labels.ravel())
    keep = sizes >= min_area
    keep[0] = False
    return keep[labels]


class Block(nn.Module):
    def __init__(self, a, b):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(a, b, 3, padding=1, bias=False),
            nn.BatchNorm2d(b),
            nn.ReLU(),
            nn.Conv2d(b, b, 3, padding=1, bias=False),
            nn.BatchNorm2d(b),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class ScratchUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.ModuleList(
            [Block(1, 24), Block(24, 48), Block(48, 96), Block(96, 192)]
        )
        self.dec = nn.ModuleList([Block(288, 96), Block(144, 48), Block(72, 24)])
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Conv2d(24, 4, 1)

    def forward(self, x):
        skips = []
        for i, b in enumerate(self.enc):
            x = b(x)
            skips.append(x)
            if i < 3:
                x = self.pool(x)
        for skip, b in zip(skips[-2::-1], self.dec):
            x = b(
                torch.cat(
                    [
                        nn.functional.interpolate(
                            x,
                            size=skip.shape[-2:],
                            mode="bilinear",
                            align_corners=False,
                        ),
                        skip,
                    ],
                    dim=1,
                )
            )
        return self.head(x)


def make_model(name, pretrained=True):
    if name == "scratch_unet":
        return ScratchUNet()
    import segmentation_models_pytorch as smp

    if name == "resnet34_unet":
        return smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet" if pretrained else None,
            in_channels=1,
            classes=4,
        )
    if name == "efficientnet_b0_fpn":
        return smp.FPN(
            encoder_name="efficientnet-b0",
            encoder_weights="imagenet" if pretrained else None,
            in_channels=1,
            classes=4,
            decoder_pyramid_channels=128,
            decoder_segmentation_channels=64,
        )
    raise ValueError(name)


Writing /kaggle/working/final_pipeline.py


In [3]:
%%writefile /kaggle/working/predict_final.py
"""Load a trained final-pipeline checkpoint and predict consistent masks.

This module is shared by the Kaggle inference notebook and optional Gradio app.
"""

from pathlib import Path
import json
import numpy as np
from PIL import Image
import torch
from final_pipeline import CLASSES, H, W, make_model, remove_components, encode_rle


class Segmenter:
    def __init__(self, checkpoint, selection, device=None):
        self.device = torch.device(
            device
            or (
                "cuda"
                if torch.cuda.is_available()
                else "mps" if torch.backends.mps.is_available() else "cpu"
            )
        )
        self.selection = json.loads(Path(selection).read_text())
        checkpoint = Path(checkpoint)
        names = self.selection.get("model_names", [self.selection["model"]])
        self.models = []
        for name in names:
            if len(names) > 1 and not checkpoint.is_dir():
                raise ValueError("An ensemble requires a directory of checkpoints")
            if checkpoint.is_dir():
                nested = checkpoint / name / "best.pt"
                path = nested if nested.is_file() else checkpoint / f"{name}.pt"
            else:
                path = checkpoint
            saved = torch.load(path, map_location=self.device, weights_only=False)
            if saved["architecture"] != name:
                raise ValueError("Checkpoint and selection architecture disagree")
            model = make_model(name, pretrained=False).to(self.device)
            model.load_state_dict(saved["state_dict"])
            model.eval()
            self.models.append(model)

    @torch.inference_mode()
    def predict_batch(self, images, tta=None):
        if tta is None:
            tta = self.selection.get("tta", False)
        arrays = []
        for image in images:
            if isinstance(image, (str, Path)):
                with Image.open(image) as f:
                    a = np.asarray(f.convert("L"), dtype=np.float32) / 255
            else:
                raw = np.asarray(image)
                integer_input = np.issubdtype(raw.dtype, np.integer)
                a = raw.astype(np.float32)
                if a.ndim == 3:
                    rgb = a if integer_input or a.max() > 1 else a * 255
                    a = (
                        np.asarray(
                            Image.fromarray(
                                np.rint(rgb).clip(0, 255).astype("uint8")
                            ).convert("L"),
                            dtype=np.float32,
                        )
                        / 255
                    )
                elif integer_input or a.max() > 1:
                    a = a / 255
            if a.shape != (H, W):
                raise ValueError(
                    f"Expected an original {H}x{W} strip; got {a.shape}. Resizing would distort the measurement geometry."
                )
            arrays.append((a - 0.5) / 0.25)
        x = torch.from_numpy(np.stack(arrays)[:, None]).to(self.device)
        probabilities = []
        for model in self.models:
            p = model(x).sigmoid()
            if tta:
                p = (p + torch.flip(model(torch.flip(x, [-1])).sigmoid(), [-1])) / 2
            # Validation stores each model's probabilities as float16 before averaging.
            probabilities.append(p.float().cpu().numpy().astype(np.float16))
        p = (
            sum(value.astype(np.float32) for value in probabilities)
            / len(probabilities)
        ).astype(np.float16)
        masks = np.empty_like(p, dtype=bool)
        for i in range(len(p)):
            for c in range(4):
                masks[i, c] = remove_components(
                    p[i, c] >= self.selection["thresholds"][c],
                    self.selection["min_areas"][c],
                )
        return masks, p

    def rles(self, images):
        masks, _ = self.predict_batch(images)
        return [
            {c: encode_rle(mask) for c, mask in zip(CLASSES, item)} for item in masks
        ]


Writing /kaggle/working/predict_final.py


In [4]:
sys.path.insert(0, "/kaggle/working")
from predict_final import Segmenter
from final_pipeline import CLASSES, H, W

segmenter = Segmenter(ASSETS, selections[0])


## Predict and preserve the required sample order
Each test image is evaluated once per selected forward pass. The four masks are RLE encoded in column-major order; empty predictions use a single space. Validate row IDs, classes, shape and RLE bounds before saving.

In [5]:
data = next(
    p.parent
    for p in ROOT.rglob("sample_submission.csv")
    if (p.parent / "test").is_dir()
)
sample = pd.read_csv(data / "sample_submission.csv", keep_default_na=False)
ids = sample.image_id.unique()
predictions = {}
start = time.perf_counter()
for offset in range(0, len(ids), 16):
    batch = ids[offset : offset + 16]
    paths = [
        data / "test" / (name if name.endswith(".jpg") else name + ".jpg")
        for name in batch
    ]
    encoded = segmenter.rles(paths)
    for name, values in zip(batch, encoded):
        for label, rle in values.items():
            if rle.strip():
                runs = np.asarray(rle.split(), dtype=np.int64).reshape(-1, 2)
                assert (runs[:, 0] >= 1).all() and (runs[:, 1] > 0).all()
                assert (runs.sum(axis=1) - 1 <= H * W).all()
            predictions[(name, label)] = rle
    if offset % 512 == 0:
        print(offset, "/", len(ids), flush=True)
submission = sample.copy()
submission["mask_rle"] = [
    predictions[(r.image_id, r.defect_type)] for r in sample.itertuples()
]
assert len(submission) == 12000 and not submission.isna().any().any()
assert submission.drop(columns="mask_rle").equals(sample.drop(columns="mask_rle"))
assert set(submission.defect_type) == set(CLASSES)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Rows:", len(submission), "seconds:", time.perf_counter() - start)
print("Nonempty masks:", submission.mask_rle.str.strip().ne("").sum())
display(submission.head(8))


0 / 3000
512 / 3000
1024 / 3000
1536 / 3000
2048 / 3000
2560 / 3000
Rows: 12000 seconds: 455.6414957820001
Nonempty masks: 4370


,ID,image_id,defect_type,mask_rle
0,1,WAFNKCRR7Q,defect_a,
1,2,WAFNKCRR7Q,defect_b,
2,3,WAFNKCRR7Q,defect_c,
3,4,WAFNKCRR7Q,defect_d,
4,5,YZXKEWSUDD,defect_a,
5,6,YZXKEWSUDD,defect_b,
6,7,YZXKEWSUDD,defect_c,
7,8,YZXKEWSUDD,defect_d,
